# Parser for PubMed Abstracts

Generates JSONL with metadata + `text_to_embed` (title + abstract).

In [ ]:
from __future__ import annotations

import json
import logging
import re
from pathlib import Path


LOGGER = logging.getLogger("input_parser")

try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

INPUT_PATH = HERE /"data/abstract-semaglutid-set.txt"
OUTPUT_PATH = HERE / "preprocessing_output/semaglutide_pubmed.jsonl"
LOGGER_PATH = HERE / "logs/input_parser.log"

NOISE_BLOCK_HEADERS = {
    "Author information:",
    "Comment in",
    "Collaborators:",
    "Conflict of interest statement",
    "Erratum in",
    "Retraction in",
    "Retraction of",
    "Updated by",
    "Update in",
}

END_HEADERS = (
    "DOI:",
    "PMID:",
    "PMCID:",
    "Copyright",
    "Publication types",
    "MeSH Terms",
    "Substances",
)

ABSTRACT_LABEL_RE = re.compile(r"^[A-Z][A-Z /\-]{2,}:\s*")
DOI_RE = re.compile(r"\b10\.\d{4,9}/\S+", re.IGNORECASE)
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")


In [7]:
def split_records(text: str) -> list[str]:
    """Split the raw input into numbered PubMed record blocks.

    Args:
        text: Raw input containing numbered records (e.g., "1. ", "2. ").

    Returns:
        A list of record blocks. Returns an empty list when no delimiters exist.
    """
    starts = [m.start() for m in re.finditer(r"(?m)^\d+\.\s", text)]
    if not starts:
        return []
    starts.append(len(text))
    records = []
    for i in range(len(starts) - 1):
        records.append(text[starts[i] : starts[i + 1]].strip())
    return records


def split_sections_with_ranges(lines: list[str]) -> list[tuple[int, int]]:
    """Return index ranges for contiguous, non-empty line sections.

    Args:
        lines: Lines from a single record block.

    Returns:
        A list of (start, end) index pairs for each contiguous section.
    """
    ranges = []
    start = None
    for i, ln in enumerate(lines):
        if ln.strip() == "":
            if start is not None:
                ranges.append((start, i - 1))
                start = None
            continue
        if start is None:
            start = i
    if start is not None:
        ranges.append((start, len(lines) - 1))
    return ranges


def parse_record(block: str) -> dict:
    """Parse a single PubMed record block into structured fields.

    Args:
        block: Raw text for one PubMed record (including citation lines).

    Returns:
        A dictionary with parsed metadata and abstract text.
    """
    try:
        lines = [ln.rstrip() for ln in block.splitlines()]

        # Strip the leading record number (e.g., "1. ") if present.
        if lines and re.match(r"^\d+\.\s", lines[0]):
            lines[0] = re.sub(r"^\d+\.\s", "", lines[0], count=1)

        ranges = split_sections_with_ranges(lines)

        def block_from_range(idx: int):
            # Helper to fetch a section by index, returning (lines, end_index).
            if len(ranges) > idx:
                s, e = ranges[idx]
                return lines[s : e + 1], e + 1
            return [], 0

        citation_block, citation_end = block_from_range(0)
        title_block, title_end = block_from_range(1)
        authors_block, authors_end = block_from_range(2)

        citation = " ".join([ln.strip() for ln in citation_block]).strip() or None
        title = " ".join([ln.strip() for ln in title_block]).strip() or None

        year = None
        journal = None
        if citation:
            ym = YEAR_RE.search(citation)
            if ym:
                year = ym.group(0)
                journal = citation[: ym.start()].strip().rstrip(".")
            else:
                journal = citation.strip().rstrip(".")

        pmid = None
        doi = None
        for ln in lines:
            if ln.startswith("PMID:"):
                pmid = re.sub(r"\D", "", ln)
            if doi is None:
                dm = DOI_RE.search(ln)
                if dm:
                    doi = dm.group(0).rstrip(".;")

        abstract_lines = []
        in_noise_block = False
        in_abstract = False

        # Start after the citation/title/authors sections when available.
        start_idx = authors_end or title_end or citation_end or 0

        for i in range(start_idx, len(lines)):
            raw = lines[i]
            line = raw.strip()

            if not line:
                if in_noise_block:
                    in_noise_block = False
                elif in_abstract:
                    # Preserve paragraph breaks inside the abstract.
                    abstract_lines.append("")
                continue

            if any(line.startswith(h) for h in NOISE_BLOCK_HEADERS):
                in_noise_block = True
                continue

            if line.startswith(END_HEADERS):
                # Stop when the abstract has ended and metadata sections begin.
                if in_abstract:
                    break
                continue

            if in_noise_block:
                continue

            if not in_abstract:
                # Allow for labeled or unlabeled abstracts.
                if ABSTRACT_LABEL_RE.match(line):
                    in_abstract = True
                    abstract_lines.append(line)
                    continue
                in_abstract = True
                abstract_lines.append(line)
                continue

            abstract_lines.append(line)

        abstract = "".join(abstract_lines).strip()

        # Combine title + abstract for embedding input.
        text_to_embed = title or ""
        if abstract:
            text_to_embed = f"{text_to_embed}{abstract}" if text_to_embed else abstract

        return {
            "pmid": pmid,
            "doi": doi,
            "year": year,
            "journal": journal,
            "title": title,
            "abstract": abstract,
            "text_to_embed": text_to_embed,
        }
    except Exception:
        LOGGER.exception("Failed to parse record block")
        raise


In [9]:
text = INPUT_PATH.read_text(errors="ignore")
records = split_records(text)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

parsed = [parse_record(r) for r in records]
filtered = [r for r in parsed if r.get("abstract")]
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for rec in filtered:
        f.write(json.dumps(rec, ensure_ascii=True) + "")

missing_pmid = sum(1 for r in parsed if not r.get("pmid"))
missing_doi = sum(1 for r in parsed if not r.get("doi"))
missing_abs = sum(1 for r in parsed if not r.get("abstract"))

logger = LOGGER
if not logger.handlers:
    formatter = logging.Formatter("%(levelname)s:%(name)s:%(message)s")
    handler = logging.StreamHandler()
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

    LOGGER_PATH.parent.mkdir(parents=True, exist_ok=True)
    file_handler = logging.FileHandler(LOGGER_PATH)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

logger.info("Records: %s", len(parsed))
logger.info("Missing PMID: %s", missing_pmid)
logger.info("Missing DOI: %s", missing_doi)
logger.info("Missing abstract: %s", missing_abs)
logger.info("Written (non-empty abstract): %s", len(filtered))
logger.info("Output: %s", OUTPUT_PATH)


INFO:input_parser:Records: 1663
INFO:input_parser:Missing PMID: 86
INFO:input_parser:Missing DOI: 23
INFO:input_parser:Missing abstract: 91
INFO:input_parser:Written (non-empty abstract): 1572
INFO:input_parser:Output: /Users/ftzavellos/Law_and_Tech/drug_explanations/Code/preprocessing_output/semaglutide_pubmed.jsonl
